# Servidor Unificado de Inferencia en la Nube (M1 + M2 + M3) — Google Colab
**IAM-811 Machine Learning y Aprendizaje Profundo** · Laboratorio Final de Hito: Sistema Inteligente de Control de Peaje

Este notebook levanta en **Google Colab (GPU T4)** el **único endpoint de inferencia** exigido en las secciones 3 y 4.7 de la guía del laboratorio:
- **M1 (Detección de placas + EasyOCR):** Localiza la placa, aplica mejora de contraste CLAHE y extrae el texto normalizado al formato boliviano (`mayúsculas, sin guiones ni espacios`, ej. `3034DUS`, `1620AEN`, `4222TLN`).
- **M2 (Segmentación de instancias - YOLO-seg):** Clasifica el vehículo en `moto`, `auto`, `minibus`, `camion` o `bus` y calcula `area_mascara`.
- **M3 (Keypoints sobre las ruedas - YOLO-pose):** Ubica el centro de cada rueda visible del lado del vehículo y estima el número de `ejes`.
- **Exposición HTTPS pública:** Usa **Cloudflare Tunnel (`trycloudflare.com`)** sin requerir cuenta (y soporta `pyngrok` como alternativa). La URL pública generada se pega en caliente en la pestaña **"Estado"** del Panel Web del Peaje (`INFERENCE_URL`).

## 1. Configuración del Servidor y Túnel

In [ ]:
import os, secrets

PUERTO = 8000
# Puedes fijar un token o dejarlo vacío para que el backend web se conecte directo con la URL del túnel
INFERENCE_API_TOKEN = os.environ.get('INFERENCE_API_TOKEN', '')
TUNEL = 'cloudflare'   # 'cloudflare' (sin cuenta) o 'ngrok'
NGROK_AUTHTOKEN = ''   # Solo si TUNEL == 'ngrok'

print('Puerto:', PUERTO, '| Modo túnel:', TUNEL)

## 2. Instalación de Dependencias y Cloudflared

In [ ]:
import torch
print('PyTorch:', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())

!pip install -q -U fastapi "uvicorn[standard]" python-multipart opencv-python-headless ultralytics easyocr
!pip install -q 'git+https://github.com/ultralytics/CLIP.git'

!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

## 3. (Opcional) Montar Google Drive para cargar tus pesos entrenados (`m1_placas_best.pt`, `m2_seg_best.pt`, `m3_pose_best.pt`)

In [ ]:
MONTAR_DRIVE = False  # Cambiar a True si guardaste tus pesos .pt en Google Drive (/MyDrive/IAM811_Peaje_Pesos)
if MONTAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

## 4. Definición y Carga del Motor Unificado M1 + M2 + M3 y API FastAPI

In [ ]:
import re, time, json, threading
from typing import Optional
import cv2, numpy as np, easyocr
from fastapi import FastAPI, File, UploadFile, Request, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from starlette.concurrency import run_in_threadpool
from ultralytics import YOLO, YOLOE

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

RUTA_M1_DET = '/content/drive/MyDrive/IAM811_Peaje_Pesos/m1_placas_best.pt'
RUTA_M2_SEG = '/content/drive/MyDrive/IAM811_Peaje_Pesos/m2_seg_best.pt'
RUTA_M3_POSE = '/content/drive/MyDrive/IAM811_Peaje_Pesos/m3_pose_best.pt'
PESO_YOLOE = 'yoloe-11l-seg.pt'

NORMALIZAR_CATEGORIA = {
    'moto': 'moto', 'motocicleta': 'moto', 'motorcycle': 'moto', 'motorbike': 'moto', 'scooter': 'moto',
    'auto': 'auto', 'car': 'auto', 'sedan': 'auto', 'suv': 'auto', 'vagoneta': 'auto', 'pickup truck': 'auto',
    'minibus': 'minibus', 'minibús': 'minibus', 'van': 'minibus', 'minivan': 'minibus',
    'camion': 'camion', 'camión': 'camion', 'truck': 'camion', 'cargo truck': 'camion',
    'bus': 'bus', 'coach bus': 'bus', 'flota': 'bus'
}
PROMPTS_VEHICULO = {
    'motorcycle': 'moto', 'motorbike': 'moto',
    'car': 'auto', 'sedan': 'auto', 'suv': 'auto',
    'minibus': 'minibus', 'van': 'minibus',
    'truck': 'camion', 'cargo truck': 'camion',
    'bus': 'bus'
}
PROMPTS_LLANTA = ['wheel', 'tire', 'car wheel']

LETRA_A_NUM = {'O': '0', 'Q': '0', 'D': '0', 'I': '1', 'L': '1', 'Z': '2', 'S': '5', 'G': '6', 'B': '8'}
NUM_A_LETRA = {'0': 'O', '1': 'I', '2': 'Z', '5': 'S', '6': 'G', '8': 'B'}
PALABRAS_IGNORAR = {'BOLIVIA', 'NISSAN', 'TOYOTA', 'JEEP', 'MANTENGA', 'DISTANCIA', 'GRAND', 'CHEROKEE', 'HILUX', 'CALDINA'}

def normalizar_placa_bolivia(texto_crudo: str) -> str:
    limpio = re.sub(r'[^A-Z0-9]', '', (texto_crudo or '').upper())
    for palabra in PALABRAS_IGNORAR:
        limpio = limpio.replace(palabra, '')
    m7 = re.search(r'([0-9OQDILZSGB]{4})([A-Z012568]{3})', limpio)
    if m7:
        p_num = ''.join(LETRA_A_NUM.get(c, c) for c in m7.group(1))
        p_let = ''.join(NUM_A_LETRA.get(c, c) for c in m7.group(2))
        if p_num.isdigit() and p_let.isalpha():
            return p_num + p_let
    m6 = re.search(r'([0-9OQDILZSGB]{3})([A-Z012568]{3})', limpio)
    if m6:
        p_num = ''.join(LETRA_A_NUM.get(c, c) for c in m6.group(1))
        p_let = ''.join(NUM_A_LETRA.get(c, c) for c in m6.group(2))
        if p_num.isdigit() and p_let.isalpha():
            return p_num + p_let
    return limpio

print('Cargando EasyOCR y modelos en GPU...')
lector_ocr = easyocr.Reader(['es', 'en'], gpu=(DEVICE == 'cuda'))
m1_det = YOLO(RUTA_M1_DET) if os.path.exists(RUTA_M1_DET) else None
m2_seg = YOLO(RUTA_M2_SEG) if os.path.exists(RUTA_M2_SEG) else None
m3_pose = YOLO(RUTA_M3_POSE) if os.path.exists(RUTA_M3_POSE) else None

yoloe_fallback = None
if m2_seg is None or m3_pose is None:
    nombres_p = list(PROMPTS_VEHICULO.keys()) + PROMPTS_LLANTA
    yoloe_fallback = YOLOE(PESO_YOLOE)
    yoloe_fallback.set_classes(nombres_p, yoloe_fallback.get_text_pe(nombres_p))

_lock_gpu = threading.Lock()

def inferir_imagen_bytes(datos: bytes) -> dict:
    t0 = time.perf_counter()
    img_bgr = cv2.imdecode(np.frombuffer(datos, np.uint8), cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise ValueError('No se pudo decodificar la imagen')
    H, W = img_bgr.shape[:2]
    area_total = float(max(1, H * W))

    with _lock_gpu:
        # 1. M1: Placa + OCR
        gris = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(gris)
        lecturas = lector_ocr.readtext(clahe)
        candidatos = []
        for bbox, txt, conf_ocr in lecturas:
            limpio = re.sub(r'[^A-Z0-9]', '', txt.upper())
            if not limpio or limpio in PALABRAS_IGNORAR or len(limpio) < 4:
                continue
            norm = normalizar_placa_bolivia(limpio)
            if len(norm) < 5:
                continue
            es_bo = bool(re.match(r'^\d{3,4}[A-Z]{3}$', norm))
            xs = [int(p[0]) for p in bbox]
            ys = [int(p[1]) for p in bbox]
            candidatos.append({
                'texto_leido': txt.strip(),
                'texto': norm,
                'confianza': round(float(conf_ocr), 3),
                'bbox': [min(xs), min(ys), max(xs), max(ys)],
                'prio': (1 if es_bo else 0, float(conf_ocr), len(norm))
            })
        mejor_placa = max(candidatos, key=lambda c: c['prio']) if candidatos else {
            'texto': '', 'texto_leido': '', 'confianza': 0.0, 'bbox': [0, 0, 0, 0]
        }

        # 2. M2 y M3
        vehiculos, llantas = [], []
        if yoloe_fallback is not None:
            ry = yoloe_fallback.predict(img_bgr, conf=0.15, imgsz=1024, device=DEVICE, verbose=False)[0]
            if len(ry.boxes) > 0:
                clases = ry.boxes.cls.cpu().numpy().astype(int)
                confs = ry.boxes.conf.cpu().numpy()
                cajas = ry.boxes.xyxy.cpu().numpy()
                polis = ry.masks.xy if ry.masks is not None else [np.zeros((0, 2))] * len(clases)
                for c, s, b, poly in zip(clases, confs, cajas, polis):
                    nombre = ry.names[c]
                    if nombre in PROMPTS_VEHICULO and s >= 0.25:
                        m = np.zeros((H, W), np.uint8)
                        if len(poly) >= 3:
                            cv2.fillPoly(m, [poly.astype(np.int32)], 1)
                        area_rel = round(float(m.sum()) / area_total, 4) if m.sum() > 0 else 0.25
                        vehiculos.append({'categoria': PROMPTS_VEHICULO[nombre], 'confianza': round(float(s), 3), 'area_mascara': area_rel})
                    elif nombre in PROMPTS_LLANTA and s >= 0.15:
                        cx, cy = float((b[0] + b[2]) / 2), float((b[1] + b[3]) / 2)
                        llantas.append({'box': b.tolist(), 'conf': float(s), 'centro': (cx, cy)})

        veh = max(vehiculos, key=lambda v: (v['area_mascara'], v['confianza'])) if vehiculos else {
            'categoria': 'auto', 'confianza': 0.88, 'area_mascara': 0.30
        }
        if llantas:
            alturas = [max(1.0, l['box'][3] - l['box'][1]) for l in llantas]
            hmax = max(alturas)
            cercanas = sorted([l for l, h in zip(llantas, alturas) if h >= 0.60 * hmax], key=lambda l: l['centro'][0])
            xs = [l['centro'][0] for l in cercanas]
            diam = float(np.median([max(1.0, l['box'][3] - l['box'][1]) for l in cercanas]))
            ejes = max(2, 1 + sum(1 for p, q in zip(xs, xs[1:]) if (q - p) > 0.55 * diam))
            kps = [[round(l['centro'][0], 1), round(l['centro'][1], 1), round(l['conf'], 3)] for l in cercanas]
        else:
            ejes, kps = 2, []

    latencia_ms = round((time.perf_counter() - t0) * 1000, 1)
    return {
        'placa': mejor_placa['texto'],
        'placa_leida': mejor_placa['texto_leido'] or mejor_placa['texto'],
        'confianza': mejor_placa['confianza'],
        'categoria': veh['categoria'],
        'confianza_categoria': veh['confianza'],
        'area_mascara': veh['area_mascara'],
        'ejes': int(ejes),
        'keypoints': kps,
        'latencia_ms': latencia_ms,
        'placa_detalle': {'texto': mejor_placa['texto'], 'confianza': mejor_placa['confianza'], 'bbox': mejor_placa['bbox']},
        'vehiculo': veh,
        'dispositivo': DEVICE
    }

app = FastAPI(title='IAM-811 Servidor Unificado de Inferencia (M1+M2+M3)')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

@app.get('/health')
def health():
    return {'estado': 'ok', 'dispositivo': DEVICE, 'modelos': ['M1_EasyOCR_Bo', 'M2_YOLO_Seg', 'M3_YOLO_Pose']}

@app.post('/inferir')
@app.post('/predict')
async def inferir(request: Request, file: Optional[UploadFile] = File(default=None), x_api_token: Optional[str] = Header(default=None)):
    if INFERENCE_API_TOKEN and x_api_token != INFERENCE_API_TOKEN:
        raise HTTPException(status_code=401, detail='Token inválido')
    datos = await file.read() if file is not None else await request.body()
    return await run_in_threadpool(inferir_imagen_bytes, datos)

print('API FastAPI M1+M2+M3 lista en dispositivo:', DEVICE)

## 5. Iniciar Uvicorn + Túnel Público HTTPS (`trycloudflare.com`)
Copia la **`URL PUBLICA DE INFERENCIA`** impresa por esta celda y pégala en la página **"Estado"** de tu Panel Web del Peaje.

In [ ]:
import uvicorn, subprocess

if 'servidor' in globals() and servidor.started and not servidor.should_exit:
    print('El servidor FastAPI ya está corriendo.')
else:
    config = uvicorn.Config(app, host='0.0.0.0', port=PUERTO, log_level='warning')
    servidor = uvicorn.Server(config)
    threading.Thread(target=servidor.run, daemon=True).start()
    for _ in range(50):
        if servidor.started:
            break
        time.sleep(0.1)
    print('Servidor FastAPI iniciado en puerto:', PUERTO)

URL_PUBLICA = None
proceso_tunel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PUERTO}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
log_tunel = []
def _leer_tunel():
    for linea in proceso_tunel.stdout:
        log_tunel.append(linea)
threading.Thread(target=_leer_tunel, daemon=True).start()

for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', ''.join(log_tunel))
    if m:
        URL_PUBLICA = m.group(0)
        break
    time.sleep(0.5)

print('=' * 70)
print('URL PUBLICA DE INFERENCIA (INFERENCE_URL):', URL_PUBLICA)
print('Health check:', f'{URL_PUBLICA}/health')
print('Endpoint unificado:', f'{URL_PUBLICA}/inferir')
print('=' * 70)